# Lab 4: Writing the Description Prompt

**Workshop 2, block 3. 20 minutes.**

Whatever the description does not mention is unavailable to retrieval.
This prompt sets the upper limit on your Level 3 and Level 5 scores.

1. Write your prompt
2. Run it over the five images and **read the output before indexing**
3. Revise and run again
4. Index the descriptions and re-score the Lv3 questions
5. Swap prompts with the team next to you

Step 2 is the one people skip. If you cannot answer the test questions
from the descriptions, neither can your index.

> **Before you start:** `data/chroma` from lab 2, `images/` from the workshop folder, and ideally `data/images.json` from lab 3.
>
> **When you finish:** `data/descriptions.json`, and image chunks added to `data/chroma` tagged `kind="image"`. Lab 5 filters on that tag.

---
## Setup

Run these two cells first. They are identical in every lab, so each
notebook works on its own.

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops. **Do not commit a notebook with a key
visible in its output.**

In [3]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

chat    OK      Unified
vision  FAILED  NotFoundError: Error code: 404 - {'error': {'type': 'invalid_request_error', 'code': 'DeploymentNotFound', 'message': 'The AP
embed   OK      1536 dimensions


In [46]:
# ---- helpers ------------------------------------------------------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# One PersistentClient per path, cached for the life of this kernel.
# chromadb caches internal state per path, so deleting the folder and
# opening a fresh PersistentClient while an earlier one from this same
# session is still alive corrupts the connection: you get "attempt to
# write a readonly database" or "database is locked" on the very next
# call. Rebuilding your index more than once per session, which the
# "change one setting, re-run" loop asks you to do, hits this every
# time with the naive version.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P

    if path not in _stores:
        # First time this path is opened in this session. Safe to wipe a
        # stale, wrong-chromadb-version index here, since no client for
        # this path exists in this process yet.
        if reset and _P(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by "
                f"a different chromadb version. Delete that folder and rebuild, or "
                f"install the pinned version from requirements.txt."
            ) from None

    client = _stores[path]
    if reset:
        # Reset now means delete-and-recreate the COLLECTION on the same
        # client, not delete-and-recreate the DIRECTORY under it. This is
        # what actually avoids the readonly/locked error on every rebuild
        # after the first.
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

helpers loaded


---
## The five fields

| Field | Ask for | Why |
|---|---|---|
| Location | The room or area, using the name on the signage | Lets a question naming a room reach the image |
| Objects, with counts | "Eleven chairs", never "some chairs" | Counting questions need this |
| Visible text, word for word | Every sign, poster and label, copied exactly | The Lv5 example asks for three exact words |
| Spatial relations | An LED wall on the north side, facing the entrance | Lv5 asks about arrangement |
| Colours and materials | Only where they identify something specific | Disambiguates similar rooms |

**Field 3 is the one to get right.** Tell the model to copy text exactly
rather than summarise, and to write `[unreadable]` rather than guess.

### Test questions

You judge a description prompt by whether its output can answer
questions. The next cell **provides three**, one each for counts, text
and layout, the fields that decide most Lv3 and Lv5 scores. The
slide-title question is a real trap: an earlier test run read "RAG" as
"PAG".

**Writing your own** (for Step 5, or later for your repository):

1. **Answerable only from the photo.** If the answer is also in the
   scraped text, you are testing retrieval, not your description.
2. **One short answer you can check:** a number, exact words, or a named
   object. "What does the room look like?" cannot be checked.
3. **One field per question**, so a miss tells you which instruction
   in your prompt to fix.
4. **Look at the photo and write the answer down before you run the
   prompt.** Otherwise you end up grading the model against itself.

In [47]:
# The five Inno Wing photos are provided in images/ next to this notebook.
IMAGES = sorted(str(p) for p in Path("images").iterdir()
                if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert len(IMAGES) >= 5, (
    f"Found {len(IMAGES)} images in images/. This lab needs the five provided photos."
)

# Three test questions about the provided photos, one per field that
# decides most scores. Each answer is visible in a photo and appears
# nowhere in the text corpus, so only a good description can answer it.
# Add your own below these (see "Writing your own" above).
TEST_QUESTIONS = [
    {"field": "count",
     "question": "How many cushions are on the long low bench in front of the tiered seating in the brainstorming area?",
     "answer": "3"},
    {"field": "text",
     "question": "What is the title of the slide on the large screen in the makerspace?",
     "answer": "Building RAG with Image Data"},
    {"field": "layout",
     "question": "What is directly to the right of the tiered seating in the brainstorming area?",
     "answer": "glass doors"},
]

print(len(IMAGES), "images")
for p in IMAGES:
    print("  ", Path(p).name)

5 images
   Brainstorming aerial.JPG
   Makerspace close up.JPG
   Open Event .JPG
   Open event.JPG
   makerspace.JPG


---
## Step 1: Write your prompt

Write instructions, not a request for a description. Compare the two
below before writing your own.

In [48]:
PROMPT_A = "Describe this image."

PROMPT_B = """List, in this order:
1. the room or area name shown on any sign
2. every object present, with a count
3. all visible text, copied exactly as written
4. what faces what, and what is next to what
5. colours and materials, only where distinctive

Write [unreadable] for text you cannot make out.
Do not summarise. Do not add anything not visible in the image."""

# ---- your prompt -------------------------------------------------
DESCRIPTION_PROMPT = """
Create a factual, searchable description of the whole image. Inspect the full frame, including the edges and background. 

Write the answer in English using exactly these headings and order:
Location: Try your best to infer the location of this area using signs and write your answer. If it's too hard and you can't think of any answer, write [unknown].
Objects: List every object and its counts. Make sure you cover every item in the image and don't miss even a single one. Create a dictionary of all the names of the objects you discovered and their corresponding counts. Before searching, set the dictionary to empty. Scan the entire picture and if you detect a new item that you haven't seen before, identify whether its name is recorded in the list. If its name is recorded, add the name's count in the dictionary by one. If not, add its name to the dictionary and set its count to 1. Write your answer using the dictionary but don't print your dictionary. Count 3 times, start all ovevr again from count 1 if your new count differs from previous ones. Write each object as 'Name: count' on its own line. Do not add item numbers, bullets, labels, explanations, or any other text; for example, write 'Laptops: 5', not '1. Laptops: 5'.
Visible text: Copy readable text exactly, preserve capitalization. Identify as much texts as possible. Use [unreadable] for texts that you think are completely unidentifiable. Do not infer missing words.
Spatial relations: Describe the positions and relationships between objects in your previous dictionary. Cover as much positions and relationships as possible. These notes do not change the object counts.
Colours & materials: Include only visible, distinguishing details. Record as much as possible.
"""

---
## Step 2: Run it and read the output

Before indexing anything. First the two example prompts on the same photo,
one cell each, then your prompt over all five photos, also one cell each —
so no single cell's output gets long enough to hide.

> **Output cut off?** A good description is long, and editors hide long
> output by default. **VS Code:** click *"View as a scrollable element"*
> under the cell, or raise *Notebook › Output: Text Line Limit* in
> Settings. **JupyterLab:** right-click the output and choose
> *"Enable Scrolling for Outputs"*. Do not skip the reading: it is the
> whole of this step.

In [48]:
print("PROMPT A\n" + "-" * 60)
print(describe_image(IMAGES[0], PROMPT_A))

PROMPT A
------------------------------------------------------------
A bright, modern interior common area or lounge with tiered seating and colorful modular furniture. In the center-left are rows of low cube seats in red, orange and yellow. Behind them is a stepped platform of bench seating in layered shades of blue and gray, with a small sign on the wall reading "ENGINEERING INNOVATION." To the right are long benches with teal cushions and a glass-walled room or entrance. The left wall has low cabinets, a small dispenser or station and several posted notices. Overhead lights and a pale carpeted floor complete the clean, open space.


In [49]:
print("PROMPT B\n" + "-" * 60)
print(describe_image(IMAGES[0], PROMPT_B))
print("\nSame model, same image, same cost. Only the prompt changed.")

PROMPT B
------------------------------------------------------------
1. Room/area name: [unreadable]

2. Objects present:
   - Seating cubes: 8 (4 red, 2 yellow, 2 orange)
   - Steps: 6 (multicolored)
   - Counter: 1
   - Water dispenser: 1
   - Shelf with printed materials: 1
   - Sign: 1 

3. Visible text:
   - "ENGINEERING"
   - "INNOVATION"
   - "NOTICE" (on the door)
   - [unreadable] (on the materials shelf)

4. Orientation:
   - Counter faces the seating area.
   - Steps are behind the seating cubes.
   - The door is next to the steps on the right side.

5. Distinctive colors and materials:
   - Cubes: Red, yellow, orange
   - Steps: Multicolored (blue, green, white, pink)
   - Counter: Light wood
   - Floor: Light gray with a textured pattern

Same model, same image, same cost. Only the prompt changed.


In [50]:
assert "TODO" not in DESCRIPTION_PROMPT, (
    "DESCRIPTION_PROMPT is still the placeholder. Write your prompt in Step 1 "
    "first. Otherwise the model answers the word 'TODO' instead of describing "
    "the image, and those replies are what gets indexed."
)

# One cell per photo below, not a loop in one cell: five descriptions in a
# single cell's output is exactly what gets hidden or cut off. Each cell
# fills in the same `descriptions` dict.
descriptions = {}

In [51]:
path = IMAGES[0]
descriptions[path] = describe_image(path, DESCRIPTION_PROMPT)
print(path)
print("-" * 70)
print(descriptions[path])

images\Brainstorming aerial.JPG
----------------------------------------------------------------------
**Location:** [unknown]

**Objects:** 
- Stairs: 1
- Cubes (various colors): 8 (3 red, 2 yellow, 3 orange)
- Table/benches: 3
- Wall panels: 2
- Sign: 1
- Water dispenser: 1

**Spatial relations:** 
- The stairs are positioned to the back/right of the image, leading up. 
- The cubes are arranged in a row, placed in front of the stairs and to the left.
- The table/benches are in front of the stairs and to the right of the cubes.
- The wall panels are vertical elements on the back wall, with one green and one white.
- The sign indicating "ENGINEERING INNOVATION" is mounted on the wall above the stairs.

**Colours & materials:** 
- The cubes are colored red, yellow, and orange. 
- The stairs are made of a mix of brown and blue materials.
- The walls are painted in green and white tones. 
- The floor appears to be a grey carpet with a linear texture. 
- The water dispenser is metallic.


In [52]:
path = IMAGES[1]
descriptions[path] = describe_image(path, DESCRIPTION_PROMPT)
print(path)
print("-" * 70)
print(descriptions[path])

images\Makerspace close up.JPG
----------------------------------------------------------------------
### Location:
[unknown]

### Objects:
1. Laptops: 5
2. People: 6
3. Table: 5
4. Whiteboard: 1
5. Monitor: 2
6. Cables: 2
7. Trash bin: 1
8. Black chair: 6
9. Plant: 3

### Visible text:
Building PNG with brave data  
Add title  
Digital Fabrication Lab 2, Green Center  
Click the image above to start the interactive experience.  
...  
[unreadable]

### Spatial relations:
- The tables are arranged in an open space with a clear path in between.
- There are 5 tables, with laptops placed on each, facing towards a presentation area.
- The six people are gathered around one of the tables towards the center of the image, actively engaged in discussion.
- Two monitors are displayed on a countertop against the back wall, near the whiteboard.
- The whiteboard is positioned against the wall to the right of the group of people.
- Cables are visible connecting various devices, resting on the table

In [52]:
path = IMAGES[2]
descriptions[path] = describe_image(path, DESCRIPTION_PROMPT)
print(path)
print("-" * 70)
print(descriptions[path])

images\Open Event .JPG
----------------------------------------------------------------------
1. Location:
[unknown zone]

2. Objects (with counts):
- grey plastic chairs (individual, not in the stack): 20
- grey plastic chairs in a stacked pile on a dolly/trolley: 12
- modular/hex/triangular white tables on casters (grouped learning tables): 6
- rectangular white tables on casters (smaller separate tables): 2
- tall storage cabinet (floor-standing): 1
- metal shelving / large rack structure at right: 1
- suspended model airplane (ceiling-hung): 1
- freestanding white flip-chart / whiteboard on stand: 1
- wall-mounted printed posters/signs (cluster on left wall): 4
- ceiling rectangular light panels: 11
- glass wall / glass doors with metal handles (right side): 1 set
- textured metal floor plate/threshold panel at left doorway: 1
- grey striped carpeted floor / rug: 1
- laptops in use on tables: 3
- backpacks / bags on tables or floor: 2
- small printed flyers/sheets placed on tables 

In [41]:
path = IMAGES[3]
descriptions[path] = describe_image(path, DESCRIPTION_PROMPT)
print(path)
print("-" * 70)
print(descriptions[path])

images\Open event.JPG
----------------------------------------------------------------------
Location: Open Event Area

Objects: 9 white modular tables (various shapes), 38 grey plastic chairs, 1 stack of approximately 10 grey plastic chairs leaning against left wall, 6 grey plastic chairs lined up against left wall, 1 long serpentine white table assembly in foreground (comprised of 3 connected modules), 1 tall grey metal storage cabinet (rear center-left), 1 large metal shelving unit with multiple grey storage bins (rear right), 1 hanging model airplane suspended from ceiling (rear right), 1 orange cable reel on a table (mid‑right), 2 yellow safety helmets on a rear table (rear right), 1 black mobile partition on casters (right foreground), 1 red fire extinguisher mounted on the black partition (right foreground), 1 freestanding poster/stand with printed panels (rear center-right), multiple wall posters and notices on left wall (clustered), 1 small wall sign with circular prohibition 

In [42]:
path = IMAGES[4]
descriptions[path] = describe_image(path, DESCRIPTION_PROMPT)
print(path)
print("-" * 70)
print(descriptions[path])

images\makerspace.JPG
----------------------------------------------------------------------
Location: [unknown zone]

Objects: 8 light‑wood high work tables with white/grey metal legs; 22 dark round wooden stools with black metal frames; 5 open laptop computers on central tables; 2 large wall‑mounted display monitors on the right side; 1 freestanding white flip‑chart board on the right side; 1 multi‑port power strip on the foreground table; 2 printed sticker sheets on table surfaces (one on the foreground table, one on a right‑side table); 5 human people; 1 full‑height glass partition wall running across the rear of the room; 1 red wall‑mounted fire extinguisher in a white case on the left wall; 1 stack of blue plastic storage bins on the far right back counter; 1 long row of white lower storage cabinets under the right windows; 1 floor power cable running from foreground table to floor

Visible text: On right‑hand large display monitor (right side near windows): Building RAG Image Da

In [43]:
# Can each test question be answered from the descriptions alone?
# If the answer is not in the text, retrieval cannot find it later.
for q in TEST_QUESTIONS:
    print(f"[{q['field']}] {q['question']}")
    print(f"   expected: {q['answer']}")
    if q["answer"].strip().isdigit():
        # A bare digit matches almost any text, so check counts by eye.
        print("   a count: find it by eye in the descriptions above")
    else:
        where = [Path(p).name for p, d in descriptions.items()
                 if answer_present(q["answer"], [{"text": d}])]
        print("   found in:", ", ".join(where) if where else "NOWHERE. Revise your prompt.")
    print()

[count] How many cushions are on the long low bench in front of the tiered seating in the brainstorming area?
   expected: 3
   a count: find it by eye in the descriptions above

[text] What is the title of the slide on the large screen in the makerspace?
   expected: Building RAG with Image Data
   found in: Makerspace close up.JPG, makerspace.JPG

[layout] What is directly to the right of the tiered seating in the brainstorming area?
   expected: glass doors
   found in: Brainstorming aerial.JPG, makerspace.JPG



---
## Step 3: Revise

Most teams need two or three attempts. The first is almost always too
general. Edit `DESCRIPTION_PROMPT` and re-run.

| Prompts that scored | Prompts that did not |
|---|---|
| Asked for counts explicitly | Asked to "describe this image" |
| Asked for text to be copied exactly | Asked for a summary or caption |
| Named the room or area | Did not mention text at all |
| Said what to do when unreadable | Left the model to decide what mattered |

---
## Step 4: Cache, index, re-score

**Cache by image path.** Without it you re-describe every image each
time you change something downstream, and experimenting becomes slow
enough that you stop.

Descriptions go into the same store as your scraped text, as ordinary
chunks. No separate pipeline. Tag them `kind="image"` so lab 5 can
filter on it.

In [44]:
Path("data").mkdir(exist_ok=True)
Path("data/descriptions.json").write_text(json.dumps(descriptions, indent=1))
print(f"cached {len(descriptions)} descriptions")

store = get_store("data/chroma", name="workshop")   # same index as lab 2

# Remove the previous run's image chunks first. The ids are reused
# (img_0, img_1, ...), and Chroma will not overwrite an existing id with
# add(), so without this a revised prompt never reaches the index.
store.delete(where={"kind": "image"})

add_to_store(
    store,
    texts=list(descriptions.values()),
    metadatas=[{"url": p, "kind": "image"} for p in descriptions],
    ids=[f"img_{i}" for i in range(len(descriptions))],
)
print(f"indexed. store now holds {store.count()} chunks")

cached 5 descriptions
indexed. store now holds 5 chunks


In [45]:
# Success looks like: an `image` chunk from the right photo at or near the
# top, and its text contains the answer. The Lv3 poster question should
# still miss. Its poster is not one of these five photos (Cause A in lab 3).
dev = load_dev_set("dev_set.json")
checks = ([(q["question"], q["answer"]) for q in TEST_QUESTIONS] +
          [(d["question"], d["answer"]) for d in dev if d["level"] in (3, 5)])

for question, expected in checks:
    hits = query(store, question, k=3)
    print(question)
    print(f"   expected: {expected}")
    for c in hits:
        kind = c["metadata"].get("kind", "text")
        print(f"   {c['distance']:.3f}  {kind:<5}  {c['metadata'].get('url', '?')[:50]}")
    print()

How many cushions are on the long low bench in front of the tiered seating in the brainstorming area?
   expected: 3
   0.883  image  images\Brainstorming aerial.JPG
   1.085  image  images\Open Event .JPG
   1.096  image  images\Open event.JPG

What is the title of the slide on the large screen in the makerspace?
   expected: Building RAG with Image Data
   0.995  image  images\Open event.JPG
   1.039  image  images\Makerspace close up.JPG
   1.050  image  images\Open Event .JPG

What is directly to the right of the tiered seating in the brainstorming area?
   expected: glass doors
   0.953  image  images\Brainstorming aerial.JPG
   1.015  image  images\Open event.JPG
   1.049  image  images\Open Event .JPG

What were the winning teams in Pitch New Tech Ideas in 2025?
   expected: Smartsocks, Digitio, Gingtrolley
   1.594  image  images\Brainstorming aerial.JPG
   1.672  image  images\Open event.JPG
   1.695  image  images\Makerspace close up.JPG

On the wall at the back of the brains

---
## Step 5: Swap with the team next to you

Run their prompt over the same five images. Which questions does each
prompt answer that the other misses?

Two teams with different prompts get different Lv3 scores from identical
images. The prompt is the whole difference.

### Going further

- **Two prompts per image.** One for objects and layout, one for text
  only. The model attends to whatever you emphasise.
- **Split large images.** A poster downscaled to fit the input loses its
  small print. Cut it into overlapping tiles and describe each.
- **Run OCR alongside.** A vision model paraphrases; an OCR engine
  transcribes character by character. Tesseract and PaddleOCR both run
  locally, so both are permitted.